# Play Bridge Crossing

Choose an environment with the buttons, preview an `rgb_array` frame inline, then run the last cell to play with the arrow keys or WASD in a pygame window.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from PIL import Image
from masa.common.notebook_play import make_reset_env, sync_selected_env

VALID_ENV_NAMES = ("BridgeCrossing", "BridgeCrossingV2")

def make_env(env_name, **kwargs):
    if env_name == "BridgeCrossing":
        from masa.envs.tabular.bridge_crossing import BridgeCrossing

        return BridgeCrossing(**kwargs)
    if env_name == "BridgeCrossingV2":
        from masa.envs.tabular.bridge_crossing_v2 import BridgeCrossingV2

        return BridgeCrossingV2(**kwargs)
    raise ValueError(f"env_name must be one of {VALID_ENV_NAMES!r}")

ENV_SELECTOR = widgets.ToggleButtons(
    options=VALID_ENV_NAMES,
    value="BridgeCrossing",
    description="Env",
)
display(ENV_SELECTOR)

ENV_NAME = ENV_SELECTOR.value
SEED = 0


In [ ]:
ENV_NAME = ENV_SELECTOR.value
preview_env = make_env(ENV_NAME, render_mode="rgb_array", render_window_size=512)
obs, info = preview_env.reset(seed=SEED)
print("reset:", {"obs": obs, "info": info})
display(Image.fromarray(preview_env.render()))
preview_env.close()


In [ ]:
def _print_reset(obs, info):
    print("reset:", {"obs": obs, "info": info})


def play_env(env_name=None, seed=SEED):
    import pygame

    follow_selector = env_name is None
    if follow_selector:
        env_name = ENV_SELECTOR.value
    action_keys = {
        pygame.K_LEFT: 0,
        pygame.K_a: 0,
        pygame.K_RIGHT: 1,
        pygame.K_d: 1,
        pygame.K_DOWN: 2,
        pygame.K_s: 2,
        pygame.K_UP: 3,
        pygame.K_w: 3,
        pygame.K_SPACE: 4,
    }
    env, obs, info = make_reset_env(make_env, env_name, seed=seed, render_mode="human", render_window_size=512)
    finished = False
    print("Controls: arrows or WASD move, Space waits, R resets, Q or Escape quits.")
    _print_reset(obs, info)

    try:
        running = True
        while running and not env.human_window_closed:
            if follow_selector:
                env, env_name, obs, info, switched = sync_selected_env(
                    env,
                    env_name,
                    ENV_SELECTOR,
                    make_env,
                    seed=seed,
                    render_window_size=512,
                    pygame=pygame,
                )
                if switched:
                    finished = False
                    print("switched:", env_name)
                    _print_reset(obs, info)

            action = None
            for event in pygame.event.get():
                if not env.handle_pygame_event(event):
                    running = False
                    break
                if event.type != pygame.KEYDOWN:
                    continue
                if event.key in (pygame.K_q, pygame.K_ESCAPE):
                    running = False
                    break
                if event.key == pygame.K_r:
                    obs, info = env.reset(seed=seed)
                    finished = False
                    _print_reset(obs, info)
                    continue
                if event.key in action_keys and not finished:
                    action = action_keys[event.key]

            if action is None:
                env.render()
                continue

            obs, reward, terminated, truncated, info = env.step(action)
            finished = terminated or truncated
            print({
                "obs": obs,
                "reward": reward,
                "terminated": terminated,
                "truncated": truncated,
                "info": info,
            })
    finally:
        env.close()

play_env(seed=SEED)
